In [2]:
import sqlite3
import os

DB_PATH = "db/bible.db"

def match_persons_to_words():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()

    # 1️⃣ 从 persons 表中取出 name_en 和对应的 person_key
    cur.execute("""
        SELECT TRIM(name_en), person_key
        FROM persons
        WHERE name_en IS NOT NULL AND person_key IS NOT NULL
    """)
    persons = cur.fetchall()

    print(f"✅ Loaded {len(persons)} persons from persons table")

    total_updated = 0

    # 2️⃣ 按顺序遍历每一个 person
    for name_en, person_key in persons:
        name_en_clean = name_en.strip().lower()

        # 3️⃣ 去 words 表里找 word 匹配（忽略大小写）
        cur.execute("""
            SELECT id
            FROM words
            WHERE LOWER(TRIM(word)) = ?
              AND (entity_key IS NULL OR entity_key = '')
        """, (name_en_clean,))

        rows = cur.fetchall()

        # 4️⃣ 找到就回填 person_key
        for (row_id,) in rows:
            cur.execute("""
                UPDATE words
                SET entity_key = ?
                WHERE id = ?
            """, (person_key, row_id))
            total_updated += 1

    conn.commit()
    conn.close()

    print(f"✅ Updated {total_updated} rows in words.entity_key")

if __name__ == "__main__":
    match_persons_to_words()

✅ Loaded 49 persons from persons table
✅ Updated 86 rows in words.entity_key
